# Integrated biodiversity-evidence corpus preparation

**Purpose:** Create the reusable publication-level corpus and abstract sidecar for substantive results.

The grain is exactly one screening-eligible publication (`UT`). Screening metadata is joined one-to-one with driver, Threats L0, geography, ecosystem realm, study-design/method, and current taxonomic group fields. List labels are normalized but never exploded. Empty geography lists become `Not Applicable`; any geography list containing `Unclear` becomes exactly `Unclear`; and country lists are validated against the shared World Bank economy mapping. No direction, year, threat, realm, geography, study, or taxa filter is applied here; every downstream notebook owns its analytical subset.

In [1]:
import sys
from pathlib import Path

import pandas as pd

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data_helpers").is_dir() and (candidate / "checklists").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the biodiversity repository root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_helpers import results_config
from data_helpers.prep import corpus
from data_helpers.prep import evidence_corpus_prep
from data_helpers.prep import taxa_analysis_prep
from data_helpers._config import get_merged_corpus_config


In [2]:
RESULTS_CONFIG = results_config.load_results_config()
SCREENING_CONFIG = RESULTS_CONFIG["screening_analysis_prep"]
EVIDENCE_CONFIG = RESULTS_CONFIG["biodiversity_evidence_prep"]
INCOME_CONFIG = RESULTS_CONFIG["world_bank_income"]
TAXA_CONFIG = RESULTS_CONFIG["taxa_analysis_prep"]
CORPUS_CONFIG = get_merged_corpus_config()
WORLD_BANK_MAPPING = PROJECT_ROOT / INCOME_CONFIG["world_bank_mapping"]
SCREENING_STORE = evidence_corpus_prep.ScreeningPreparedStore(PROJECT_ROOT / SCREENING_CONFIG["output_directory"])
EVIDENCE_STORE = evidence_corpus_prep.BiodiversityEvidenceStore(PROJECT_ROOT / EVIDENCE_CONFIG["output_directory"])
TAXA_MAPPING = taxa_analysis_prep.load_taxa_mapping(PROJECT_ROOT / TAXA_CONFIG["taxa_group_mapping"])
TAXA_STORE = taxa_analysis_prep.TaxaPreparedStore(PROJECT_ROOT / TAXA_CONFIG["output_directory"])


## 1. Join every coding dimension without changing the eligible UT grain

The eligible screening artifact supplies the left-hand key order. Every configured coding source and the prepared taxa artifact must contain the identical `UT` set and one row per key. The three existing geography columns are standardized in place; assertions protect the publication count, `UT` order, and complete column contract.

In [3]:
eligible_screening, screening_manifest = SCREENING_STORE.load_eligible()
merged = corpus.build_merged_corpus_from_eligible_screening(eligible_screening)
taxa_bundle = TAXA_STORE.load(mapping=TAXA_MAPPING)
publications, abstracts = evidence_corpus_prep.build_biodiversity_evidence_corpus(merged, taxa_bundle.articles)
original_uts = publications["UT"].copy()
original_columns = publications.columns.tolist()
geography_audit = evidence_corpus_prep.standardize_geography_lists(
    publications, world_bank_mapping_path=WORLD_BANK_MAPPING
)
assert len(publications) == len(original_uts)
assert publications["UT"].equals(original_uts)
assert publications.columns.tolist() == original_columns
print(f"Integrated grain: {len(publications):,} rows / {publications['UT'].nunique():,} unique UTs / {len(publications.columns):,} columns.")
display(geography_audit.style.format({"changed_values": "{:,}"}).hide(axis="index"))
display(publications["s2_dir"].value_counts(dropna=False).rename("n_publications").rename_axis("direction").reset_index().style.format({"n_publications": "{:,}"}).hide(axis="index"))


Integrated grain: 605,199 rows / 605,199 unique UTs / 39 columns.


column,rule,changed_values
pred_countries,contains_unclear_to_unclear,212
pred_countries,empty_to_not_applicable,"40,495"
pred_countries,non_world_bank_country_to_unclear,"3,693"
pred_regions,empty_to_not_applicable,390
pred_subregions,empty_to_not_applicable,"6,120"


direction,n_publications
negative,"253,081"
mixed,"141,267"
positive,"112,684"
unclear,"98,167"


## 2. Write and verify the integrated corpus and abstract sidecar

The 40-column main Parquet retains list labels, standard taxonomy ranks, deterministic taxonomy summaries, and typed nested matches with full lineage. Abstracts remain lossless in a separate two-column Parquet; `wos_categories` is intentionally not retained. The manifest records all columns, source signatures, direction counts, upstream schemas, and the taxonomic grouping-rule fingerprint.

In [4]:
sources = {
    "eligible_screening_artifact": SCREENING_STORE.eligible_path,
    "corpus_config": PROJECT_ROOT / SCREENING_CONFIG["corpus_config"],
    "world_bank_country_mapping": WORLD_BANK_MAPPING,
    "prepared_taxa_artifact": TAXA_STORE.article_path,
    "prepared_taxa_matches": TAXA_STORE.match_path,
    "prepared_taxa_manifest": TAXA_STORE.manifest_path,
}
sources.update({f"coding_{source['name']}": source["path"] for source in CORPUS_CONFIG["coding_sources"]})
manifest = evidence_corpus_prep.build_biodiversity_manifest(
    publications, abstracts, sources=sources, screening_manifest=screening_manifest, taxa_manifest=taxa_bundle.manifest, repository_root=PROJECT_ROOT
)
bundle = evidence_corpus_prep.BiodiversityEvidenceBuild(publications, abstracts, TAXA_STORE.match_path, manifest)
written = EVIDENCE_STORE.write(bundle)
reloaded = EVIDENCE_STORE.load(columns=["UT", *evidence_corpus_prep.GEOGRAPHY_LIST_COLUMNS])
reloaded_abstracts = EVIDENCE_STORE.load_abstracts()
assert reloaded.publications["UT"].equals(publications["UT"])
assert len(reloaded.publications) == len(publications)
for column in evidence_corpus_prep.GEOGRAPHY_LIST_COLUMNS:
    assert reloaded.publications[column].equals(publications[column])
assert reloaded_abstracts["UT"].equals(abstracts["UT"])
print("Verified integrated corpus:")
for path in written:
    print(f"  {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size / 1024**2:.2f} MiB)")
if not EVIDENCE_CONFIG["retain_eligible_build_cache"]:
    removed = SCREENING_STORE.remove_eligible_build_cache()
    if removed:
        print(f"Removed consumed build cache: {SCREENING_STORE.eligible_path.relative_to(PROJECT_ROOT)}")


Verified integrated corpus:
  notebooks/data_processing/outputs/04_biodiversity_evidence_corpus_prep/biodiversity_evidence_corpus.parquet (150.23 MiB)
  notebooks/data_processing/outputs/04_biodiversity_evidence_corpus_prep/biodiversity_evidence_abstracts.parquet (362.02 MiB)
  notebooks/data_processing/outputs/04_biodiversity_evidence_corpus_prep/manifest.json (0.00 MiB)
Removed consumed build cache: notebooks/data_processing/outputs/03_screening_analysis_prep/eligible_screening_publications.parquet


## Handoff contract

Results notebooks request only their required columns from `biodiversity_evidence_corpus.parquet` through `BiodiversityEvidenceStore`. The prepared geography contract preserves list-valued columns: empty lists are `Not Applicable`, lists containing `Unclear` are reduced to `Unclear`, and `pred_countries` contains only special labels or country lists fully accepted by the shared World Bank economy lookup. Analyses that need text explicitly join `biodiversity_evidence_abstracts.parquet` by `UT`. They may filter or explode labels only after loading, and must establish a one-row-per-UT denominator before constructing an exploded attribution table.